In [1]:
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from mlflow import MlflowClient
from dotenv import load_dotenv
import pickle
import pathlib
import pandas as pd
from datetime import datetime
from sklearn.feature_extraction import DictVectorizer
import math
import optuna
from optuna.samplers import TPESampler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error
mlflow.sklearn.autolog()

In [3]:
load_dotenv(override=True)  # Carga las variables del archivo .env
EXPERIMENT_NAME = "/Users/aissafosado@gmail.com/nyc-taxi-experiments"

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

In [4]:
def read_dataframe(path):
    df = pd.read_parquet(path)
    df["duration"] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime).dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    df[["PULocationID", "DOLocationID"]] = df[["PULocationID", "DOLocationID"]].astype(str)
    return df

In [5]:
df_train = read_dataframe('../data/green_tripdata_2025-01.parquet')
df_val = read_dataframe('../data/green_tripdata_2025-02.parquet')

df_train["PU_DO"] = df_train["PULocationID"] + "_" + df_train["DOLocationID"]
df_val["PU_DO"] = df_val["PULocationID"] + "_" + df_val["DOLocationID"]

Feature Engineering + One Hot Encoding

In [6]:
def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)

In [7]:
# Dictionaries for preprocessing
dv = DictVectorizer()

# Define categorical and numerical variables
categorical = ['PU_DO']
numerical = ['trip_distance']

# Fit DictVectorizer on training data
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

# Validation
X_val = preprocess(df_val, dv)

In [8]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [9]:
training_dataset = mlflow.data.from_numpy(X_train.data, targets=y_train, name="green_tripdata_2025-01")
validation_dataset = mlflow.data.from_numpy(X_val.data, targets=y_val, name="green_tripdata_2025-02")

## Random Forest

In [10]:
# ------------------------------------------------------------
# Definir la función objetivo para Optuna
#    - Recibe un `trial`, que se usa para proponer hiperparámetros.
#    - Entrena un modelo con esos hiperparámetros.
#    - Calcula la métrica de validación (RMSE) y la retorna (Optuna la minimizará).
#    - Abrimos un run anidado de MLflow para registrar cada trial.
# ------------------------------------------------------------

def objective(trial: optuna.trial.Trial):
    # Hiperparámetros MUESTREADOS por Optuna en CADA trial.
    # Nota: usamos log=True para emular rangos log-uniformes (similar a loguniform).
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 30, 200),
        "max_depth": trial.suggest_int("max_depth", 4, 150),
        "min_samples_split": trial.suggest_int("min_samples_split", 20, 200),
        "max_features": trial.suggest_int("max_features", 50, 120),
        "ccp_alpha": trial.suggest_float("ccp_alpha",   math.exp(-4), math.exp(-2), log=True),
        "random_state": 42,                      
    }

    # Run anidado para dejar rastro de cada trial en MLflow
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "randomforest")  # etiqueta informativa
        mlflow.log_params(params)                  # registra hiperparámetros del trial

        # Entrenamiento con el conjunto de validación
        rf = RandomForestRegressor(**params)
        rf.fit(X_train, y_train)

        # Predicción y métrica en validación
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        # Registrar la métrica principal
        mlflow.log_metric("rmse", rmse)

        # La "signature" describe la estructura esperada de entrada y salida del modelo:
        # incluye los nombres, tipos y forma (shape) de las variables de entrada y el tipo de salida.
        # MLflow la usa para validar datos en inferencia y documentar el modelo en el Model Registry.
        signature = infer_signature(X_val, y_pred)

        # Guardar el modelo del trial como artefacto en MLflow.
        mlflow.sklearn.log_model(
            sk_model = rf,
            name="model",
            input_example=X_val[:5],
            signature=signature,
        )

    # Optuna minimiza el valor retornado
    return rmse

In [11]:
mlflow.sklearn.autolog(log_models=False)

# ------------------------------------------------------------
# Crear el estudio de Optuna
#    - Usamos TPE (Tree-structured Parzen Estimator) como sampler.
#    - direction="minimize" porque queremos minimizar el RMSE.
# ------------------------------------------------------------
sampler = TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)

# ------------------------------------------------------------
# Ejecutar la optimización (n_trials = número de intentos)
#    - Cada trial ejecuta la función objetivo con un set distinto de hiperparámetros.
#    - Abrimos un run "padre" para agrupar toda la búsqueda.
# ------------------------------------------------------------
with mlflow.start_run(run_name="RandomForest Hyperparameter Optimization (Optuna)", nested=True):
    study.optimize(objective, n_trials=10)

    # --------------------------------------------------------
    # Recuperar y registrar los mejores hiperparámetros
    # --------------------------------------------------------
    best_params = study.best_params
    # Asegurar tipos/campos fijos (por claridad y consistencia)
    best_params["max_depth"] = int(best_params["max_depth"])
    best_params["seed"] = 42
    best_params["objective"] = "reg:squarederror"

    mlflow.log_params(best_params)

    # Etiquetas del run "padre" (metadatos del experimento)
    mlflow.set_tags({
        "project": "NYC Taxi Time Prediction Project",
        "optimizer_engine": "optuna",
        "model_family": "randomforest",
        "feature_set_version": 1,
    })

    # --------------------------------------------------------
    # 7) Entrenar un modelo FINAL con los mejores hiperparámetros
    #    (normalmente se haría sobre train+val o con CV; aquí mantenemos el patrón original)
    # --------------------------------------------------------

    # Select parameters
    rf = RandomForestRegressor(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        min_samples_split=best_params["min_samples_split"],
        max_features=best_params["max_features"],
        ccp_alpha=best_params["ccp_alpha"],
        random_state=42
    )
    # Fit the model
    rf.fit(X_train, y_train)

    # Evaluar y registrar la métrica final en validación
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    # --------------------------------------------------------
    # 8) Guardar artefactos adicionales (p. ej. el preprocesador)
    # --------------------------------------------------------
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    # La "signature" describe la estructura esperada de entrada y salida del modelo:
    # incluye los nombres, tipos y forma (shape) de las variables de entrada y el tipo de salida.
    # MLflow la usa para validar datos en inferencia y documentar el modelo en el Model Registry.
    # Si X_val es la matriz dispersa (scipy.sparse) salida de DictVectorizer:
    feature_names = dv.get_feature_names_out()
    input_example = pd.DataFrame(X_val[:5].toarray(), columns=feature_names)

    # Para que las longitudes coincidan, usa el mismo slice en y_pred
    signature = infer_signature(input_example, y_val[:5])

    # Guardar el modelo del trial como artefacto en MLflow.
    mlflow.sklearn.log_model(
    sk_model=rf,                    # Trained RandomForestRegressor
    name="model",                   # Folder inside MLflow run to store the model
    input_example= input_example,   # First few rows of validation data
    signature=signature,            
)

[I 2025-11-25 22:51:13,001] A new study created in memory with name: no-name-90af7070-78d9-4925-babd-693014ca40e5


2025/11/25 22:51:47 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run grandiose-pug-587 at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134/runs/160c318599a24cbbb96d39e8d97ae455
🧪 View experiment at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134


[I 2025-11-25 22:51:54,629] Trial 0 finished with value: 5.66617993971921 and parameters: {'n_estimators': 94, 'max_depth': 143, 'min_samples_split': 152, 'max_features': 92, 'ccp_alpha': 0.025022928883219303}. Best is trial 0 with value: 5.66617993971921.


2025/11/25 22:52:10 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-25 22:52:13,371] Trial 1 finished with value: 8.280677057897906 and parameters: {'n_estimators': 56, 'max_depth': 12, 'min_samples_split': 176, 'max_features': 92, 'ccp_alpha': 0.07548246929868654}. Best is trial 0 with value: 5.66617993971921.


...

[I 2025-11-25 22:55:42,591] Trial 8 finished with value: 5.9171376337706025 and parameters: {'n_estimators': 50, 'max_depth': 76, 'min_samples_split': 26, 'max_features': 114, 'ccp_alpha': 0.030732331451840258}. Best is trial 0 with value: 5.66617993971921.


2025/11/25 22:56:11 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-25 22:56:22,855] Trial 9 finished with value: 6.685764239078063 and parameters: {'n_estimators': 143, 'max_depth': 49, 'min_samples_split': 114, 'max_features': 88, 'ccp_alpha': 0.02650846696393051}. Best is trial 0 with value: 5.66617993971921.


🏃 View run charming-stork-216 at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134/runs/886427bb626f41c8aeaa057a99676334
🧪 View experiment at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134


c:\Users\Vivienne\apps\nyc-taxi-predictions-2025\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
2025/11/25 22:56:56 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run RandomForest Hyperparameter Optimization (Optuna) at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134/runs/b30b7a6be3c4449ca6a54dd648c67984
🧪 View experiment at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134


## GradientBoosting

In [12]:
# ------------------------------------------------------------
# Definir la función objetivo para Optuna
#    - Recibe un `trial`, que se usa para proponer hiperparámetros.
#    - Entrena un modelo con esos hiperparámetros.
#    - Calcula la métrica de validación (RMSE) y la retorna (Optuna la minimizará).
#    - Abrimos un run anidado de MLflow para registrar cada trial.
# ------------------------------------------------------------

def objective(trial: optuna.trial.Trial):
    # Hiperparámetros MUESTREADOS por Optuna en CADA trial.
    # Nota: usamos log=True para emular rangos log-uniformes (similar a loguniform).
    params = {
        "learning_rate": trial.suggest_float("learning_rate", math.exp(-2), math.exp(3), log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 20, 150),
        "min_samples_split": trial.suggest_int("min_samples_split", 50, 200),
        "max_features": trial.suggest_int("max_features", 20, 150),
        "alpha": trial.suggest_float("alpha",   math.exp(-4), math.exp(-3), log=True),
        "random_state": 42,                      
    }

    # Run anidado para dejar rastro de cada trial en MLflow
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "randomforest")  # etiqueta informativa
        mlflow.log_params(params)                  # registra hiperparámetros del trial

        # Entrenamiento con el conjunto de validación
        rf = GradientBoostingRegressor(**params)
        rf.fit(X_train, y_train)

        # Predicción y métrica en validación
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        # Registrar la métrica principal
        mlflow.log_metric("rmse", rmse)

        # La "signature" describe la estructura esperada de entrada y salida del modelo:
        # incluye los nombres, tipos y forma (shape) de las variables de entrada y el tipo de salida.
        # MLflow la usa para validar datos en inferencia y documentar el modelo en el Model Registry.
        signature = infer_signature(X_val, y_pred)

        # Guardar el modelo del trial como artefacto en MLflow.
        mlflow.sklearn.log_model(
            sk_model = rf,
            name="model",
            input_example=X_val[:5],
            signature=signature,
        )

    # Optuna minimiza el valor retornado
    return rmse

In [13]:
mlflow.sklearn.autolog(log_models=False)

# ------------------------------------------------------------
# Crear el estudio de Optuna
#    - Usamos TPE (Tree-structured Parzen Estimator) como sampler.
#    - direction="minimize" porque queremos minimizar el RMSE.
# ------------------------------------------------------------
sampler = TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)

# ------------------------------------------------------------
# Ejecutar la optimización (n_trials = número de intentos)
#    - Cada trial ejecuta la función objetivo con un set distinto de hiperparámetros.
#    - Abrimos un run "padre" para agrupar toda la búsqueda.
# ------------------------------------------------------------
with mlflow.start_run(run_name="GradientBoosting Hyperparameter Optimization (Optuna)", nested=True):
    study.optimize(objective, n_trials=10)

    # --------------------------------------------------------
    # Recuperar y registrar los mejores hiperparámetros
    # --------------------------------------------------------
    best_params = study.best_params
    # Asegurar tipos/campos fijos (por claridad y consistencia)
    best_params["max_depth"] = int(best_params["max_depth"])
    best_params["seed"] = 42
    best_params["objective"] = "reg:squarederror"

    mlflow.log_params(best_params)

    # Etiquetas del run "padre" (metadatos del experimento)
    mlflow.set_tags({
        "project": "NYC Taxi Time Prediction Project",
        "optimizer_engine": "optuna",
        "model_family": "gradientboosting",
        "feature_set_version": 1,
    })

    # --------------------------------------------------------
    # 7) Entrenar un modelo FINAL con los mejores hiperparámetros
    #    (normalmente se haría sobre train+val o con CV; aquí mantenemos el patrón original)
    # --------------------------------------------------------

    # Select parameters
    gb = GradientBoostingRegressor(
        learning_rate=best_params["learning_rate"],
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        min_samples_split=best_params["min_samples_split"],
        max_features=best_params["max_features"],
        alpha=best_params["alpha"],
        random_state=42
    )

    # Fit the model
    gb.fit(X_train, y_train)

    # Evaluar y registrar la métrica final en validación
    y_pred = gb.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    # --------------------------------------------------------
    # 8) Guardar artefactos adicionales (p. ej. el preprocesador)
    # --------------------------------------------------------
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    # La "signature" describe la estructura esperada de entrada y salida del modelo:
    # incluye los nombres, tipos y forma (shape) de las variables de entrada y el tipo de salida.
    # MLflow la usa para validar datos en inferencia y documentar el modelo en el Model Registry.
    # Si X_val es la matriz dispersa (scipy.sparse) salida de DictVectorizer:
    feature_names = dv.get_feature_names_out()
    input_example = pd.DataFrame(X_val[:5].toarray(), columns=feature_names)

    # Para que las longitudes coincidan, usa el mismo slice en y_pred
    signature = infer_signature(input_example, y_val[:5])

    # Guardar el modelo del trial como artefacto en MLflow.
    mlflow.sklearn.log_model(
    sk_model=gb,                    # Trained GradientBoostingRegressor
    name="model",                   # Folder inside MLflow run to store the model
    input_example= input_example,   # First few rows of validation data
    signature=signature,            
)

[I 2025-11-25 22:57:14,219] A new study created in memory with name: no-name-85cc0111-f20f-44ca-b5b9-b3531addc6da


2025/11/25 22:58:35 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run enthused-smelt-60 at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134/runs/37afcc00795046cdb3b71e087d94c1cc
🧪 View experiment at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134


[I 2025-11-25 22:59:20,637] Trial 0 finished with value: 5.710009045160773 and parameters: {'learning_rate': 0.88047001533197, 'n_estimators': 288, 'max_depth': 115, 'min_samples_split': 140, 'max_features': 40, 'alpha': 0.02140768135226643}. Best is trial 0 with value: 5.710009045160773.


2025/11/25 23:00:25 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-25 23:01:10,463] Trial 1 finished with value: 5.37456961347927 and parameters: {'learning_rate': 0.18094142133009145, 'n_estimators': 267, 'max_depth': 98, 'min_samples_split': 156, 'max_features': 22, 'alpha': 0.048311282772064666}. Best is trial 1 with value: 5.37456961347927.


...

[I 2025-11-25 23:06:57,295] Trial 8 finished with value: 10461.133001680042 and parameters: {'learning_rate': 2.082463143527972, 'n_estimators': 96, 'max_depth': 147, 'min_samples_split': 167, 'max_features': 143, 'alpha': 0.044816780293330145}. Best is trial 7 with value: 5.282587624761784.


2025/11/25 23:07:27 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-25 23:07:36,617] Trial 9 finished with value: 1.3606033992417057e+50 and parameters: {'learning_rate': 2.689888906482186, 'n_estimators': 281, 'max_depth': 31, 'min_samples_split': 79, 'max_features': 25, 'alpha': 0.025357780594395314}. Best is trial 7 with value: 5.282587624761784.


🏃 View run abundant-foal-48 at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134/runs/d16a2b74b5aa421eabdbceea76df682c
🧪 View experiment at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134


c:\Users\Vivienne\apps\nyc-taxi-predictions-2025\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(
2025/11/25 23:08:30 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run GradientBoosting Hyperparameter Optimization (Optuna) at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134/runs/642b369b893b4a489cf456181c5ed561
🧪 View experiment at: https://dbc-9a9988ed-c03a.cloud.databricks.com/ml/experiments/3125568759087134


## Model Comparison

In [16]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.rmse ASC"],
    output_format="list"
)

# Modelos a evaluar
models = {"RandomForest", "GradientBoosting"}

# Seleccionar las runs con el filtro
challenger_runs = [
    run for run in runs
    if run.info.run_name 
    and any(model in run.info.run_name for model in models)
]

# Obtener el mejor run
if challenger_runs:
    best_run = challenger_runs[0]  
    params = best_run.data.params

    print("Found Challenger Run:")
    print(f"Run ID: {best_run.info.run_id}")
    print(f"Model Type: {params.get('model_type')}")
    print(f"RMSE: {best_run.data.metrics.get('rmse')}")
    print(f"Params: {params}")

else:
    print("⚠️ No RandomForest or GradientBoosting runs found.")

Found Challenger Run:
Run ID: 642b369b893b4a489cf456181c5ed561
Model Type: None
RMSE: 5.282587624761784
Params: {'alpha': '0.03080950666040755', 'ccp_alpha': '0.0', 'criterion': 'friedman_mse', 'init': 'None', 'learning_rate': '0.16072549094016098', 'loss': 'squared_error', 'max_depth': '53', 'max_features': '60', 'max_leaf_nodes': 'None', 'min_impurity_decrease': '0.0', 'min_samples_leaf': '1', 'min_samples_split': '150', 'min_weight_fraction_leaf': '0.0', 'n_estimators': '278', 'n_iter_no_change': 'None', 'objective': 'reg:squarederror', 'random_state': '42', 'seed': '42', 'subsample': '1.0', 'tol': '0.0001', 'validation_fraction': '0.1', 'verbose': '0', 'warm_start': 'False'}


Register in Mlflow

In [17]:
model_name = "workspace.default.nyc-taxi-model"

result = mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/model",
    name=model_name
)

Registered model 'workspace.default.nyc-taxi-model' already exists. Creating a new version of this model...
2025/11/25 23:16:58 WARNING mlflow.tracking._model_registry.fluent: Run with id 642b369b893b4a489cf456181c5ed561 has no artifacts at artifact path 'model', registering model based on models:/m-804e196f41b847559136f629cf2ba4bf instead


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

Created version '1' of model 'workspace.default.nyc-taxi-model'.


In [18]:
# Añadir alias challenger
client = MlflowClient()

model_version = result.version
new_alias = "Challenger"

client.set_registered_model_alias(
    name=model_name,
    alias=new_alias,
    version=result.version
)

## Champion vs Challenger

In [20]:
df_test = read_dataframe('../data/green_tripdata_2025-03.parquet')

df_test["PU_DO"] = df_test["PULocationID"] + "_" + df_test["DOLocationID"]
X_test_matriz = preprocess(df_test, dv)
feature_names = dv.get_feature_names_out()

X_test = pd.DataFrame(X_test_matriz.toarray(), columns=feature_names)

y_test = df_test[target].values

XGboost (Champion)

In [22]:
model_version_uri = f"models:/{model_name}@Champion"

champion_version = mlflow.pyfunc.load_model(model_version_uri)

y_test_pred = champion_version.predict(X_test_matriz)
y_test_pred

RestException: RESOURCE_DOES_NOT_EXIST: Registered Model Alias 'champion' does not exist.